EXZECO Flood Risk Assessment - Complete Analysis Notebook
=========================================================

This notebook demonstrates the complete EXZECO workflow for preliminary
flood risk assessment using Monte Carlo simulation on Digital Elevation Models.

Based on the methodology from CEREMA:
https://www.cerema.fr/system/files/documents/2020/07/methode_exzeco_25mai2020.pdf

Author: Tobias Siegfried, hydrosolutions GmbH
Date: 2025
License: MIT

## 1. Setup and Initialization

### 1.1 Libraries
This section sets up the environment and imports necessary libraries.

In [ ]:
# 1.1 Import required libraries

# %% Import required libraries
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Add src directory to path
sys.path.append('../src')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import yaml
from typing import Tuple, Dict, List
from IPython.display import display, HTML, Image
import folium
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from tqdm.notebook import tqdm
import rasterio
from shapely.geometry import box

# Import EXZECO modules
from exzeco import ExzecoAnalysis, ExzecoConfig, load_config
from dem_utils import DEMDownloader, StudyArea
from visualization import ExzecoVisualizer

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All modules loaded successfully!")

✅ All modules loaded successfully!


### 1.2 Plotting Options

In [18]:
# 1.2 Configure high-resolution plots for better quality

# %% Configure high-resolution plots for better quality
# Set matplotlib backend for high-resolution inline plots
%matplotlib inline

# Configure matplotlib for high-resolution plots
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300  # High DPI for crisp plots
mpl.rcParams['savefig.dpi'] = 300  # High DPI for saved figures
mpl.rcParams['figure.figsize'] = [12, 8]  # Default figure size [inches]
mpl.rcParams['font.size'] = 12  # Readable font size
mpl.rcParams['axes.labelsize'] = 12  # Axis label font size
mpl.rcParams['axes.titlesize'] = 14  # Title font size
mpl.rcParams['xtick.labelsize'] = 10  # X-tick font size
mpl.rcParams['ytick.labelsize'] = 10  # Y-tick font size
mpl.rcParams['legend.fontsize'] = 11  # Legend font size

# For even better quality in VS Code notebooks, you can also use:
%config InlineBackend.figure_format = 'retina'  # High-res displays (Mac Retina, etc.)
# Alternative options: 'png', 'jpg', 'svg', 'pdf'

# Optional: Set Plotly to use high-resolution rendering
import plotly.io as pio
try:
    # Try to configure kaleido if available
    if hasattr(pio, 'kaleido') and pio.kaleido.scope is not None:
        pio.kaleido.scope.mathjax = None  # Faster rendering
except (AttributeError, TypeError):
    # Skip if kaleido is not available or configured
    pass
pio.renderers.default = "notebook"  # Ensure plots show in notebook

print("✅ High-resolution plot configuration applied!")
print(f"   • Figure DPI: {mpl.rcParams['figure.dpi']}")
print(f"   • Save DPI: {mpl.rcParams['savefig.dpi']}")
print(f"   • Default figure size: {mpl.rcParams['figure.figsize']}")
print(f"   • Backend format: retina (high-resolution)")
print(f"   • Plotly renderer: {pio.renderers.default}")

✅ High-resolution plot configuration applied!
   • Figure DPI: 300.0
   • Save DPI: 300.0
   • Default figure size: [12.0, 8.0]
   • Backend format: retina (high-resolution)
   • Plotly renderer: notebook


### 📊 **High-Resolution Plot Configuration Applied**

The notebook is now configured for high-quality inline plots with the following settings:

**Matplotlib Configuration:**
- **DPI**: 300 (both display and save) - 3x higher than default (100 DPI)
- **Figure Format**: Retina/High-DPI for crisp display on high-resolution screens
- **Default Size**: 12×8 inches for better visibility
- **Font Sizes**: Optimized for readability at high resolution

**Alternative Configuration Options:**
```python
# For different output formats (run in a cell if needed):
%config InlineBackend.figure_format = 'svg'    # Vector format, scalable
%config InlineBackend.figure_format = 'png'    # Standard bitmap
%config InlineBackend.figure_format = 'retina' # High-DPI displays (current)

# For custom DPI (run in a cell if needed):
mpl.rcParams['figure.dpi'] = 150  # Medium quality
mpl.rcParams['figure.dpi'] = 300  # High quality (current)
mpl.rcParams['figure.dpi'] = 600  # Very high quality (slower)
```

**Benefits:**
- ✅ Crisp, sharp plots especially on Retina/4K displays
- ✅ Better text readability in plots
- ✅ Professional-quality figures suitable for presentations/reports
- ✅ Consistent high quality across matplotlib, seaborn, and plotly plots

**Note:** Higher DPI may increase rendering time and memory usage, but provides significantly better visual quality.

## 2. Configuration
Load configuration settings from a YAML file. This includes study area definition, DEM source, and other parameters.
This allows for easy customization of the analysis parameters.

In [19]:
# Load configuration from config file
config = load_config('../config/config.yml')

print("Configuration:")
print(f"  Noise levels: {config.noise_levels}")
print(f"  Iterations: {config.iterations}")
print(f"  Min drainage area: {config.min_drainage_area} km²")
print(f"  Parallel jobs: {config.n_jobs}")

Configuration:
  Noise levels: [1.0, 2.0]
  Iterations: 10
  Min drainage area: 0.001 km²
  Parallel jobs: -1


## 3. Define Study Area
This section defines the study area either by loading a shapefile/geopackage or by using bounding box coordinates.

### Error Handling & Fallback Strategy
The system now includes robust error handling:

1. **Primary**: Attempts to load shapefile/geopackage from `config.shapefile_path`
2. **Validation**: Checks if file exists and contains valid geometries
3. **Fallback**: If shapefile fails, uses bounding box coordinates from `config.bounds`
4. **Error Messages**: Provides clear feedback about what went wrong and what action was taken

**Configuration Options in `config.yml`:**
```yaml
study_area:
  shapefile_path: "./data/doi/qmp_doi.gpkg"  # Primary data source
  bounds: [74.3, 42.3, 74.9, 43.2]         # Fallback coordinates [minx, miny, maxx, maxy]
```

This ensures the analysis can proceed even if the specified shapefile is missing or corrupted.

In [ ]:
# %% 3. Define study area - Configurable in config.yml

# Construct path to shapefile from config
# The path in config.yml is relative to the project root.
# The notebook is in notebooks/, so we need to go up one level.
shapefile_path = Path('../') / config.shapefile_path

# Load from shapefile or GeoJSON with error handling
try:
    # Check if the shapefile path exists
    if not shapefile_path.exists():
        raise FileNotFoundError(f"Shapefile not found at: {shapefile_path}")
    
    # Try to load the shapefile
    study_gdf = gpd.read_file(shapefile_path)
    
    if study_gdf.empty:
        raise ValueError(f"Shapefile is empty: {shapefile_path}")
    
    print(f"✅ Shapefile loaded successfully: {shapefile_path.name}")
    print(f"   • Features: {len(study_gdf)}")
    print(f"   • Geometry types: {study_gdf.geometry.type.unique()}")
    
    # Ensure the GeoDataFrame is in lat/lon coordinates (EPSG:4326)
    original_crs = study_gdf.crs
    needs_reprojection = study_gdf.crs != 'EPSG:4326'

    if needs_reprojection:
        study_gdf = study_gdf.to_crs('EPSG:4326')
        
        # Save the transformed shapefile to avoid coordinate system issues in analysis
        transformed_shapefile_path = shapefile_path.parent / f"{shapefile_path.stem}_wgs84{shapefile_path.suffix}"
        study_gdf.to_file(transformed_shapefile_path, driver='GPKG')
        print(f"🗺️  Shapefile: {shapefile_path.name} | {original_crs} → EPSG:4326 | Saved as: {transformed_shapefile_path.name}")
    else:
        transformed_shapefile_path = shapefile_path
        print(f"🗺️  Shapefile: {shapefile_path.name} | CRS: {original_crs} ✓")

    # Extract bounds and create study area
    STUDY_BOUNDS = tuple(study_gdf.total_bounds)
    
except (FileNotFoundError, ValueError, Exception) as e:
    print(f"⚠️  Error loading shapefile: {e}")
    print(f"🔄 Falling back to bounding box coordinates from config...")
    
    # Fallback to bounding box from config
    if hasattr(config, 'bounds') and config.bounds:
        STUDY_BOUNDS = tuple(config.bounds)
        print(f"📦 Using bounding box from config: {STUDY_BOUNDS}")
        
        # Create a simple rectangular polygon for the bounding box
        from shapely.geometry import box
        bbox_polygon = box(*STUDY_BOUNDS)
        study_gdf = gpd.GeoDataFrame([1], geometry=[bbox_polygon], crs='EPSG:4326')
        transformed_shapefile_path = None  # No transformed file for bounding box
        
        print(f"✅ Created study area from bounding box")
    else:
        print(f"❌ No fallback bounding box found in config!")
        raise ValueError("Neither shapefile nor bounding box coordinates are available. Please check your config.yml file.")

# Create study area and calculate area
study_area = StudyArea(STUDY_BOUNDS)
area_km2 = study_area.get_area_km2()

print(f"📍 Study Area: {area_km2:.1f} km² | Bounds: {STUDY_BOUNDS[1]:.4f}°N-{STUDY_BOUNDS[3]:.4f}°N, {STUDY_BOUNDS[0]:.4f}°E-{STUDY_BOUNDS[2]:.4f}°E")

# Create comprehensive study area visualization (static + interactive)
import contextlib
import io
import importlib
import visualization
importlib.reload(visualization)
from visualization import create_study_area_visualization

# Suppress verbose output from visualization function
print("📊 Creating study area visualizations...")
with contextlib.redirect_stdout(io.StringIO()):
    visualization_results = create_study_area_visualization(
        study_area=study_area,
        area_km2=area_km2,
        output_dir=Path('../data/outputs'),
        show_static=True,
        show_interactive=True,
        polygon_gdf=study_gdf
    )

print("✅ Study area visualizations created")

2025-09-12 15:27:38,849 - ERROR - ../data/doi/qmp_doi.gpkg: No such file or directory


DriverError: ../data/doi/qmp_doi.gpkg: No such file or directory

### ✅ **Study Area Configuration Success**

The study area has been successfully configured with the following robust error handling:

**🔧 Error Handling Features:**
- **File Validation**: Checks if shapefile exists before attempting to load
- **Geometry Validation**: Verifies that loaded data contains valid geometries  
- **CRS Handling**: Automatic coordinate system validation and reprojection to EPSG:4326
- **Fallback Strategy**: Uses bounding box coordinates if shapefile fails
- **Detailed Logging**: Clear messages about what action was taken

**📁 Data Source Used:**
- **Shapefile Path**: `{shapefile_path if 'shapefile_path' in locals() else 'N/A'}`
- **Features Loaded**: `{len(study_gdf) if 'study_gdf' in locals() else 'N/A'}`
- **Coordinate System**: `{original_crs if 'original_crs' in locals() else 'N/A'}`
- **Reprojection**: `{'Yes' if 'needs_reprojection' in locals() and needs_reprojection else 'No'}`

**🗺️ Study Area Summary:**
- **Total Area**: `{area_km2:.2f} km²` 
- **Bounds**: `{STUDY_BOUNDS[1]:.4f}°N-{STUDY_BOUNDS[3]:.4f}°N, {STUDY_BOUNDS[0]:.4f}°E-{STUDY_BOUNDS[2]:.4f}°E`

This ensures that the EXZECO analysis can proceed reliably regardless of data availability issues.

## 4. Download and Process Digital Elevation Model (DEM)

This section handles the acquisition and processing of Digital Elevation Model data for the defined study area. The DEM is the foundational dataset for all subsequent flood risk analysis in the EXZECO methodology.

### 🎯 **DEM Processing Overview**

**Objectives:**
- Download elevation data (SRTM 30m/90m resolution)
- Process and validate DEM quality
- Generate terrain visualizations

**Key Features:**
- **Smart Naming**: DEM files are automatically named after the shapefile (e.g., `qmp_doi_dem.tif`)
- **Intelligent Caching**: Checks for existing DEM files before downloading to avoid redundant downloads
- **Fallback Strategy**: SRTM1 → SRTM3 → ASTER GDEM → Synthetic DEM for testing

**Key Outputs:**
- `{shapefile_name}_dem.tif` - Processed elevation raster (named after input shapefile)
- DEM statistics and quality metrics
- Hillshade and terrain analysis plots

**Technical Notes:**
- DEM files are cached in `data/dem/cache/` directory
- Existing DEMs with matching names are reused automatically
- Handles coordinate reprojection and nodata values
- Falls back to synthetic DEM if all download sources fail

In [ ]:
# %% 4. Download and Process Digital Elevation Model (DEM)

# %% Reload EXZECO and visualization module to get the fix
import importlib
import exzeco
importlib.reload(exzeco)
from exzeco import ExzecoAnalysis, ExzecoConfig, load_config
importlib.reload(visualization)
from visualization import ExzecoVisualizer

# Reload dem_utils module to get the latest changes
import importlib
import dem_utils
importlib.reload(dem_utils)
from dem_utils import DEMDownloader

# Initialize DEM downloader
dem_downloader = DEMDownloader()

# Get absolute path for cache directory
current_dir = os.getcwd()
cache_dir = Path(current_dir).parent / "data" / "dem" / "cache"

# Generate expected DEM filename to check if it exists
expected_filename = "study_area_dem.tif"  # Default
if shapefile_path is not None:
    shapefile_stem = Path(shapefile_path).stem
    # Apply same logic as in DEM downloader
    if shapefile_stem.endswith('_wgs84'):
        shapefile_stem = shapefile_stem[:-6]
    for suffix in ['_wgs84', '_4326', '_reprojected']:
        if shapefile_stem.endswith(suffix):
            shapefile_stem = shapefile_stem[:shapefile_stem.rfind(suffix)]
            break
    expected_filename = f"{shapefile_stem}_dem.tif"

expected_dem_path = cache_dir / expected_filename
if expected_dem_path.exists():
    status = "📁 Cached"
else:
    status = "🌍 Download"

print(f"🎯 DEM Processing: {expected_filename} | Status: {status}")

# Download DEM using the refactored method with shapefile-based naming
dem_path, dem_stats = dem_downloader.download_dem_with_fallback(
    bounds=STUDY_BOUNDS,
    cache_dir=cache_dir,
    output_filename="study_area_dem.tif",  # Will be overridden if shapefile_path is provided
    product='SRTM1',  # Trying with 30m resolution
    shapefile_path=shapefile_path  # Pass shapefile path for automatic naming
)

# Read DEM data for shape information
with rasterio.open(dem_path) as src:
    dem_data = src.read(1)

print(f"✅ DEM Ready: {dem_data.shape} | {dem_stats['min_elevation']:.0f}-{dem_stats['max_elevation']:.0f}m | {dem_stats['resolution']}m")

# %% Visualize DEM using the visualization module
# Reload visualization module to get the latest DEMVisualizer class
import importlib
import visualization
importlib.reload(visualization)
from visualization import create_dem_visualization

# Create comprehensive DEM analysis visualization
dem_fig = create_dem_visualization(
    dem_path=dem_path,
    show_plot=True,
    save_individual=True,
    save_path=Path('../data/outputs/dem_analysis.png'),
    shapefile_path=transformed_shapefile_path
)

## 5. Run EXZECO Analysis

This section performs the core EXZECO (Extraction des Zones d’Écoulement) flood risk assessment analysis. 

The analysis includes:
1. **Threshold optimization**: Finding the appropriate drainage area threshold for the study area
2. **Monte Carlo simulation**: Running multiple iterations with different noise levels to generate probability maps
3. **Results generation**: Creating flood probability maps and summary statistics

The EXZECO methodology uses Monte Carlo simulation with DEM noise to assess flood risk without requiring complex hydraulic modeling.

### 5.1 EXZECO Analysis

In [ ]:
# Section 5.1: ExZeco Analysis

print("🎯 FINDING APPROPRIATE DRAINAGE AREA THRESHOLD")
print("=" * 55)

# Create temporary analyzer to analyze flow accumulation
temp_config = ExzecoConfig(
    noise_levels=[0.2, 0.4],  
    iterations=5,  
    min_drainage_area=0.001,  
    n_jobs=1  
)

temp_analyzer = ExzecoAnalysis(temp_config)
temp_analyzer.load_dem(dem_path)

# Calculate flow accumulation for original DEM
flow_dir, slopes = temp_analyzer._compute_flow_direction_d8(temp_analyzer.dem_data)
flow_acc = temp_analyzer._compute_flow_accumulation(flow_dir)

# Convert to drainage area
pixel_area_km2 = (temp_analyzer.resolution_x * temp_analyzer.resolution_y) / 1e6
drainage_area = flow_acc * pixel_area_km2

print(f"Drainage area statistics:")
print(f"  Single pixel area: {pixel_area_km2:.6f} km²")
print(f"  Min drainage: {np.min(drainage_area):.6f} km²")
print(f"  Max drainage: {np.max(drainage_area):.6f} km²")
print(f"  90th percentile: {np.percentile(drainage_area, 90):.6f} km²")
print(f"  95th percentile: {np.percentile(drainage_area, 95):.6f} km²")
print(f"  99th percentile: {np.percentile(drainage_area, 99):.6f} km²")

# Test different thresholds to find appropriate one
#thresholds_to_test = [0.01, 0.05, 0.1, 0.5]  # km²
thresholds_to_test = config.noise_levels  # Use config values
print(f"\n📊 Testing different drainage area thresholds:")
for thresh in thresholds_to_test:
    pixels_above = np.sum(drainage_area >= thresh)
    percent_above = 100 * pixels_above / drainage_area.size
    print(f"  {thresh:0.2f} km²: {pixels_above:,} pixels ({percent_above:.1f}%)")

# Choose threshold that includes 10-30% of study area (typical for EXZECO)
appropriate_threshold = 0.05  # km²
print(f"\n🎯 Selected threshold: {appropriate_threshold} km²")
print(f"   This represents {appropriate_threshold/pixel_area_km2:.0f} pixels minimum")

# Select hand-picked threshold and override automatically determined one (uncomment if needed)
appropriate_threshold = 0.1 # km²

# Configure final EXZECO analysis
final_config = ExzecoConfig(
    #noise_levels=[0.2, 0.4, 0.6, 0.8, 1.0],  
    noise_levels=config.noise_levels,  # Use config values
    iterations=config.iterations,  # Use config values  
    min_drainage_area=appropriate_threshold,
    n_jobs=config.n_jobs  # Use config values  
)

print(f"\n🚀 Running EXZECO Analysis")
print(f"   Noise levels: {final_config.noise_levels}")
print(f"   Iterations: {final_config.iterations}")  
print(f"   Min drainage area: {final_config.min_drainage_area} km²")
print(f"   Using shapefile: {transformed_shapefile_path}")

# Run complete EXZECO analysis with transformed shapefile
analyzer = ExzecoAnalysis(final_config)

try:
    results = analyzer.run_full_analysis(
        dem_path=dem_path,
        bounds=STUDY_BOUNDS,
        shapefile_path=transformed_shapefile_path  # Use the transformed shapefile
    )
    
    print(f"\n✅ EXZECO ANALYSIS COMPLETED!")
    
    # Generate summary report
    report = analyzer.generate_report()
    print(f"\n📈 ANALYSIS RESULTS:")
    display(report)
    print(f"\n🎉 EXZECO ANALYSIS SUCCESSFUL!")
    
except Exception as e:
    print(f"❌ Error in EXZECO analysis: {e}")
    import traceback
    traceback.print_exc()

### 5.2 Visualization of Results

Note, visualizations for different noise levels can be created using the `ExzecoVisualizer` class. This allows for a comprehensive analysis of flood risks across various scenarios.

In [ ]:
# Section 5.2

# %% Reload EXZECO and visualization module to get the fix
import importlib
import exzeco
importlib.reload(exzeco)
from exzeco import ExzecoAnalysis, ExzecoConfig, load_config
importlib.reload(visualization)
from visualization import ExzecoVisualizer

# Create visualizer instance
visualizer = ExzecoVisualizer(results, dem_path)

# Create flood probability visualization with shapefile overlay
fig = visualizer.plot_flood_probability_with_shapefile(
    noise_level='exzeco_200cm',
    shapefile_gdf=study_gdf,
    threshold=0.5,
    figsize=(15, 8),
    show_plot=True,
    save_path=Path('../data/outputs/flood_probability_with_boundaries.png')
)

## 6. Detect Spatial Features

This section detects spatial features such as rivers, lakes, and other water bodies within the study area.
It uses the processed DEM to identify these features, which are essential for understanding flood dynamics.

### Outputs Produced:

**1. Endorheic Basins Detection:**
   - **Hillshade visualization**: A grayscale relief map showing the terrain's 3D appearance with shading
   - **Endorheic basins overlay**: Red-colored areas indicating closed drainage basins (areas with no outlet)
   - **Area statistics**: Quantitative measurements of total endorheic area in km²
   - **Status messages**: Information about whether significant endorheic basins were detected

**2. Drainage Area Classification:**
   - **Classification map**: A color-coded visualization showing different drainage area classes
   - **Legend**: Color scale indicating drainage area thresholds (0.01, 0.05, 0.1, 0.5, 1.0 km²)
   - **Flood zone mapping**: Areas classified by their drainage characteristics and flood probability
   - **Spatial analysis**: Integration of flood probability results with drainage patterns

These visualizations help identify:
- Areas prone to water accumulation (endorheic basins)
- Drainage network patterns and flow concentration areas
- Relationship between topography and flood risk
- Spatial distribution of flood-prone zones based on drainage characteristics

In [ ]:
# Section 6
importlib.reload(visualization) # just for development
visualizer = ExzecoVisualizer(results, dem_path)

# Perform comprehensive spatial features analysis
spatial_figures = visualizer.plot_spatial_features_analysis(
    analyzer=analyzer,
    results=results,
    figsize_endorheic=(10, 8),
    figsize_drainage=(12, 8),
    shapefile_gdf=study_gdf,
    show_plots=True,
    save_paths={
        'endorheic': Path('../data/outputs/endorheic_basins_analysis.png'),
        'drainage': Path('../data/outputs/drainage_classification.png')
    },
    save_tiff=True
)

print(f"✅ Spatial features analysis completed")

## 7. Interactive Visualization
This section provides interactive visualization of the flood risk assessment results.
It allows users to explore the results of the EXZECO analysis, including flood risk maps and
spatial features detected in the study area.
This visualization aids in understanding the flood risk landscape and supports decision-making processes.

### 7.1 Map Visualization

In [ ]:
# Section 7. Visualization
importlib.reload(visualization) # just for development
visualizer = ExzecoVisualizer(results, dem_path)

# Create interactive maps and 3D visualizations of the results.

# %% Initialize visualizer
import importlib
import visualization
importlib.reload(visualization)
from visualization import ExzecoVisualizer

visualizer = ExzecoVisualizer(results, dem_path)

# %% Create interactive map
print("Creating interactive map...")

try:
    # Create map with basic features first
    interactive_map = visualizer.create_interactive_map(
        noise_level='exzeco_200cm',
        include_layers=[]  # Start with no additional layers
    )
    
    # Display map
    display(interactive_map)
    
    # Create outputs directory
    output_dir = Path("../data/outputs")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Save map
    interactive_map.save('../data/outputs/exzeco_map.html')
    print("✅ Interactive map saved to ../data/outputs/exzeco_map.html")
    
except Exception as e:
    print(f"Error creating interactive map: {e}")
    print("Creating a simple fallback map...")
    
    # Create a simple folium map as fallback
    import folium
    
    # Get center coordinates
    center_lat = (STUDY_BOUNDS[1] + STUDY_BOUNDS[3]) / 2
    center_lon = (STUDY_BOUNDS[0] + STUDY_BOUNDS[2]) / 2
    
    # Create simple map
    fallback_map = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=12,
        tiles='OpenStreetMap'
    )
    
    # Add study area boundary
    folium.Rectangle(
        bounds=[[STUDY_BOUNDS[1], STUDY_BOUNDS[0]], 
                [STUDY_BOUNDS[3], STUDY_BOUNDS[2]]],
        color='red',
        fill=True,
        fillOpacity=0.1,
        popup='Study Area'
    ).add_to(fallback_map)
    
    display(fallback_map)
    fallback_map.save('../data/outputs/exzeco_map_simple.html')
    print("✅ Simple map saved to ../data/outputs/exzeco_map_simple.html")

### 7.2 3D Visualization

In [ ]:
# %% Create 3D visualization
print("Creating 3D visualization...")

# %% Reload EXZECO and visualization module to get the fix
import importlib
import exzeco
importlib.reload(exzeco)
from exzeco import ExzecoAnalysis, ExzecoConfig, load_config
importlib.reload(visualization)
from visualization import ExzecoVisualizer
visualizer = ExzecoVisualizer(results, dem_path)

try:
    fig_3d = visualizer.create_3d_visualization('exzeco_100cm')
    fig_3d.show()
    
    # Save 3D visualization
    output_dir = Path("../data/outputs")
    output_dir.mkdir(parents=True, exist_ok=True)
    fig_3d.write_html('../data/outputs/exzeco_3d.html')
    print("✅ 3D visualization saved to ../data/outputs/exzeco_3d.html")
    
except Exception as e:
    print(f"Error creating 3D visualization: {e}")


### 7.3 Statistics

The method generates a figure containing four different subplots:

Mean Flood Probability (Top-Left): This is a line chart that plots the average flood probability for each noise level. It helps to visualize the general trend of flood risk as the noise level increases.

Flood Area Coverage (Top-Right): This is a bar chart that shows the percentage of the total area considered to be at risk. It specifically displays two categories:
- Flood Area: The percentage of the area where the flood probability is greater than 50%.
- High Risk Area: The percentage of the area where the flood probability is greater than 80%.

Risk Distribution (Bottom-Left): This subplot displays box plots for each noise level. A box plot is a standardized way of displaying the distribution of data based on a five-number summary (minimum, first quartile, median, third quartile, and maximum). This plot shows the distribution of flood probability values for all the areas that have a flood probability greater than 10%. It helps in understanding the spread, central tendency, and outliers in the risk values for each simulation.

Cumulative Risk (Bottom-Right): This is a line chart that shows the cumulative sum of the "Flood Area (%)" as the noise level increases. This can help in understanding the aggregated impact of increasing noise on the total flooded area.

In essence, this visualization provides a multi-faceted comparison of the EXZECO analysis results, enabling a user to assess the stability and sensitivity of the flood risk predictions under different simulation conditions.

In [ ]:
## 7.3 Statistics

# %% Create comparison plots
print("Creating comparison analysis...")

try:
    fig_comparison = visualizer.create_comparison_plot()
    fig_comparison.show()
    
    # Export the EXZECO Multi-Level Comparison plot
    print("\n📊 Exporting EXZECO Multi-Level Comparison plot...")
    
    # Create outputs directory if it doesn't exist
    output_dir = Path("../data/outputs")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Save as HTML (interactive)
    comparison_html_path = output_dir / "exzeco_multi_level_comparison.html"
    fig_comparison.write_html(str(comparison_html_path))
    print(f"✅ Multi-Level Comparison plot saved as HTML: {comparison_html_path}")
    
    # Save as PNG (static image) - requires kaleido
    try:
        comparison_png_path = output_dir / "exzeco_multi_level_comparison.png"
        fig_comparison.write_image(str(comparison_png_path), width=1200, height=800, scale=2)
        print(f"✅ Multi-Level Comparison plot saved as PNG: {comparison_png_path}")
    except Exception as png_error:
        print(f"⚠️  PNG export failed (requires kaleido package): {png_error}")
        print("   HTML version saved successfully - you can open it in a browser")
        
    # Optional: Save as SVG (vector format)
    try:
        comparison_svg_path = output_dir / "exzeco_multi_level_comparison.svg"
        fig_comparison.write_image(str(comparison_svg_path), format='svg', width=1200, height=800)
        print(f"✅ Multi-Level Comparison plot saved as SVG: {comparison_svg_path}")
    except Exception as svg_error:
        print(f"⚠️  SVG export failed: {svg_error}")
        
except Exception as e:
    print(f"Error creating comparison plot: {e}")

# %% Create risk heatmap
# print("Creating risk heatmap...")

# try:
#     fig_heatmap = visualizer.create_risk_heatmap('exzeco_100cm')
#     fig_heatmap.show()
# except Exception as e:
#     print(f"Error creating risk heatmap: {e}")

# %% Simple results summary
print("\n" + "="*50)
print("EXZECO ANALYSIS SUMMARY")
print("="*50)

for level, result in results.items():
    prob_map = result['probability_map']
    flooded_cells = np.sum(prob_map > 0.5)  # Cells with >50% flood probability
    total_cells = prob_map.size
    flooded_percentage = (flooded_cells / total_cells) * 100
    
    print(f"\n{level.replace('_', ' ').title()}:")
    print(f"  - Flooded area (>50% probability): {flooded_percentage:.1f}%")
    print(f"  - Max probability: {np.max(prob_map):.3f}")
    print(f"  - Mean probability: {np.mean(prob_map):.3f}")

print(f"\nAnalysis completed for study area: {STUDY_BOUNDS}")
print(f"DEM resolution: {dem_data.shape}")
print("="*50)

## 8. Interactive Dashboard


In [ ]:
# Create an interactive dashboard for exploring results.

# %% Create interactive dashboard
print("Creating interactive dashboard...")

dashboard = visualizer.create_interactive_dashboard()
display(dashboard)

# %% Export dashboard as HTML file
from dashboard_exporter import export_dashboard

# Export the dashboard with a single clean command
export_info = export_dashboard(
    dashboard=dashboard,
    output_dir=Path("../data/outputs"),
    filename="exzeco_interactive_dashboard.html",
    study_bounds=STUDY_BOUNDS,
    config=config,
    title="EXZECO Interactive Dashboard"
)

print(f"📊 Dashboard export completed: {export_info['export_method']}")
if export_info['success']:
    print(f"✅ Dashboard saved: {export_info['html_path']}")
    print(f"📂 File size: {export_info['file_size_kb']:.1f} KB")
else:
    print(f"⚠️ Export issues: {export_info.get('error', 'Unknown error')}")

## 9. Statistical Analysis

This section performs statistical analysis on the results obtained from the EXZECO methodology. It aims to derive meaningful insights from the flood risk assessment outputs.

### Key Components:

1. **Descriptive Statistics**: Calculate basic statistics (mean, median, standard deviation) for flood probability values across the study area.

2. **Spatial Distribution Analysis**: Assess the spatial distribution of flood-prone areas using kernel density estimation (KDE) and other spatial analysis techniques.

3. **Correlation Analysis**: Investigate correlations between flood risk and other variables (e.g., land use, soil type) to identify potential drivers of flood risk.

4. **Uncertainty Analysis**: Evaluate the uncertainty associated with the flood risk predictions, including the impact of DEM noise and other input parameters.

### Outputs Produced:

- **Statistical Summary**: A report summarizing the key statistics and findings from the analysis.
- **Visualizations**: Maps and charts illustrating the spatial distribution of flood risk and other relevant variables.
- **Recommendations**: Based on the analysis, provide recommendations for flood risk management and mitigation strategies.


In [ ]:
# ## 9. Statistical Analysis
# 
# Detailed statistical analysis of flood risk.

# %% Reload modules for latest updates
import importlib
import visualization
importlib.reload(visualization)
from visualization import ExzecoVisualizer

# Import risk metrics module
import sys
sys.path.append('../src')
import risk_metrics
importlib.reload(risk_metrics)
from risk_metrics import (
    compute_risk_metrics, 
    create_risk_summary_dataframe,
    analyze_risk_evolution,
    check_risk_significance,
    create_risk_visualization,
    export_risk_analysis
)

# %% Statistical analysis - Individual analysis for each noise level
print("📊 Generating individual statistical analysis for each noise level...")
print(f"   • Configuration: {config.iterations:,} iterations, {config.min_drainage_area} km² drainage threshold")
print("")

# Create a visualizer for the statistical analysis
visualizer = ExzecoVisualizer(results, dem_path=dem_path)

for level in results.keys():
    print(f"Analyzing {level}...")
    # Include config parameter and enable file saving with output directory
    visualizer.plot_statistics(
        noise_level=level, 
        config=config,  # Include config to show iterations and drainage threshold
        save_files=True,  # Enable file saving 
        output_dir=output_dir  # Save to outputs directory
    )

print(f"\n✅ Statistical analysis completed for {len(results)} noise levels")
print(f"📁 Individual analysis files saved to: {output_dir}")

# %% Comprehensive risk assessment using risk_metrics module
print(f"\n{'='*60}")
print("COMPREHENSIVE RISK ASSESSMENT")
print('='*60)

# Compute detailed risk metrics for all noise levels
print("Computing comprehensive risk metrics...")
risk_data = compute_risk_metrics(results, analyzer)

# Create structured risk summary DataFrame
risk_df = create_risk_summary_dataframe(risk_data)

# Display the risk summary
print("\n📊 Risk Assessment Summary:")
display(risk_df)

# %% Risk significance analysis
print(f"\n{'='*40}")
print("RISK SIGNIFICANCE ANALYSIS")
print('='*40)

significance = check_risk_significance(risk_data, min_threshold=0.01)

if significance['has_significant_risk']:
    print("✅ SIGNIFICANT FLOOD RISK DETECTED")
    print(f"   • Maximum flood area: {significance['max_flood_area_km2']:.3f} km²")
    print(f"   • Maximum critical risk area: {significance['max_critical_area_km2']:.3f} km²")
    print(f"   • Significant risk levels: {len(significance['significant_levels'])}/{len(results)}")
    print(f"   • Recommendation: {significance['recommendation'].replace('_', ' ').title()}")
else:
    print("⚠️  NO SIGNIFICANT FLOOD RISK DETECTED")
    print(f"   • Maximum flood area: {significance['max_flood_area_km2']:.6f} km²")
    print(f"   • Threshold used: {significance['threshold_used']} km²")
    print(f"   • Recommendation: {significance['recommendation'].replace('_', ' ').title()}")
    print("\n   Possible reasons:")
    for reason in significance['possible_reasons']:
        print(f"     - {reason}")

# %% Risk evolution analysis (if multiple noise levels)
if len(results) > 1:
    print(f"\n{'='*40}")
    print("RISK EVOLUTION ANALYSIS")
    print('='*40)
    
    evolution = analyze_risk_evolution(risk_data)
    
    if evolution['has_evolution']:
        print("📈 Risk Evolution Trends:")
        for metric, trend in evolution['trends'].items():
            change_value = evolution['changes'][metric.replace('_trend', '_change')]
            print(f"   • {metric.replace('_', ' ').title()}: {trend} (Δ {change_value:+.3f})")
        
        print(f"\n📊 Noise Level Progression:")
        for i, (level, noise_val) in enumerate(zip(evolution['sorted_levels'], evolution['noise_values'])):
            flood_area = evolution['values']['flood_areas'][i]
            print(f"   {i+1}. {level}: {noise_val:.1f}m → {flood_area:.3f} km² flood area")

# %% Create comprehensive risk visualization
print(f"\n{'='*40}")
print("GENERATING RISK VISUALIZATIONS")
print('='*40)

try:
    # Create comprehensive risk visualization
    risk_fig = create_risk_visualization(
        risk_df=risk_df,
        risk_data=risk_data,
        figsize=(15, 10),
        save_path=output_dir / 'comprehensive_risk_analysis.png'
    )
    plt.show()
    print("✅ Risk visualization created and saved")
    
except Exception as e:
    print(f"⚠️  Error creating risk visualization: {e}")

# %% Export risk analysis results
print(f"\n{'='*40}")
print("EXPORTING RISK ANALYSIS RESULTS")
print('='*40)

export_result = export_risk_analysis(
    risk_df=risk_df,
    risk_data=risk_data,
    output_dir=output_dir,
    config=config,
    formats=['csv', 'excel', 'json']
)

if export_result['success']:
    print("✅ Risk analysis results exported successfully:")
    for format_type, file_path in export_result['exported_files'].items():
        print(f"   • {format_type.upper()}: {file_path.name}")
else:
    print(f"❌ Export failed: {export_result.get('error', 'Unknown error')}")

print(f"\n✅ Comprehensive statistical analysis completed for {len(results)} noise levels")
print(f"📂 All analysis files saved to: {output_dir}")
print(f"🔧 Analysis used {config.iterations:,} iterations with {config.min_drainage_area} km² drainage threshold")

## 10. Export Results

In [ ]:
# ## 10. Export Results
# 
# Export analysis results in various formats with descriptive naming convention.
# New naming format: {file_type}_{noise_level_cm}_{iterations}_{drainage_threshold_km2}.{extension}

# Import the export function from our dedicated export module
import sys
sys.path.append('../src')  # Add src directory to path
from export import export_exzeco_results

# Export all results using the dedicated export function
export_info = export_exzeco_results(
    analyzer=analyzer,
    results=results,
    dem_path=dem_path,
    output_dir=output_dir,
    report=report,
    risk_df=risk_df,
    study_bounds=STUDY_BOUNDS,
    dem_data=dem_data
)

# Display export summary
if export_info['success']:
    print(f"\n🎉 Export completed successfully!")
    print(f"   • Total files exported: {export_info['total_files']}")
    print(f"   • Total size: {export_info['total_size_mb']:.1f} MB")
    print(f"   • Files with descriptive naming: {len(export_info['descriptive_files'])}")
    
    if export_info['errors']:
        print(f"\n⚠️  Warnings/Errors encountered:")
        for error in export_info['errors']:
            print(f"   • {error}")
else:
    print(f"\n❌ Export failed with errors:")
    for error in export_info['errors']:
        print(f"   • {error}")

## 11. Create Final Report

In [ ]:
# Generate a comprehensive HTML report using the dedicated script.

# Import the final report generation function
import sys
sys.path.append('../src')  # Add src directory to path
from write_final_report import write_final_report

# Generate the final HTML report using the dedicated script
report_result = write_final_report(
    risk_df=risk_df,
    study_bounds=STUDY_BOUNDS,
    area_km2=area_km2,
    dem_stats=dem_stats,
    config=config,
    output_dir=output_dir,
    filename='exzeco_final_report.html'
)

# Display results
if report_result['success']:
    print(f"✅ {report_result['message']}")
    print(f"📁 File size: {report_result['file_size_kb']:.1f} KB")
else:
    print(f"❌ Failed to generate report: {report_result['error']}")

In [ ]:
# ## 12. Conclusion
# 
# The EXZECO analysis is complete! 
# 
# ### Summary of Outputs:
# 1. **Raster Results**: GeoTIFF files with flood probability maps for each noise level
# 2. **Vector Results**: Shapefiles/GeoJSON with flood zones
# 3. **Interactive Map**: HTML map with multiple layers
# 4. **3D Visualization**: Interactive 3D terrain with flood zones
# 5. **Statistical Reports**: CSV/Excel files with detailed statistics
# 6. **Visualizations**: PNG/HTML charts and graphs
# 
# ### Next Steps:
# 1. Review the interactive map to identify critical flood zones
# 2. Examine the statistical reports for risk quantification
# 3. Use the vector outputs for further GIS analysis
# 4. Share the HTML report with stakeholders
# 
# ### Important Notes:
# - This is a preliminary assessment based on topography only
# - Results should be validated with field observations and historical flood data
# - For detailed planning, complement with hydraulic modeling
# - Consider local infrastructure and drainage systems not captured in the DEM

# %% Save notebook state
print("\n" + "="*60)
print("EXZECO Analysis Complete!")
print("="*60)
print(f"\nAll results saved to: {output_dir.absolute()}")
print("\nThank you for using the EXZECO Flood Risk Assessment Tool!")